# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [1]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # --- Lectura del mundo a partir de la imagen (5x6) ---
        # START en la esquina superior izquierda.
        self.start = (0, 0)

        # Estanterías / paredes (celdas no transitables).
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}

        # Piso resbaloso (dinámica distinta).
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        # Estados terminales con su recompensa R(s).
        self.terminal_states = {
            (0, 5): +10.0,   # ENTREGA
            (2, 2): +2.0,    # CARGA
            (3, 5): -10.0,   # peligro mortal
        }

        # Peligros NO terminales (penalizan pero el robot continúa).
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

        # Probabilidades segun el tipo de piso.
        # (p_intended, p_desviacion_cada_lado)
        self.normal_probs = (0.90, 0.05)
        self.slippery_probs = (0.60, 0.20)

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): un terminal es absorbente y entrega su recompensa.
        if self.is_terminal(state):
            return self.terminal_states[state]
        # Peligro no terminal: penaliza pero no termina.
        if state in self.danger_states:
            return self.danger_states[state]
        # Cualquier otra celda transitable: costo por paso.
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Terminal absorbente.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Elegimos la dinamica segun el piso del estado ACTUAL.
        if state in self.slippery_states:
            p_intended, p_perp = self.slippery_probs
        else:
            p_intended, p_perp = self.normal_probs

        # Perpendiculares a (dr, dc): giro a cada lado.
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perp),
            (perp2, p_perp),
        ]

        # Acumulamos probabilidad por estado destino (clave si varias
        # desviaciones terminan en la misma celda, p.ej. contra una pared).
        probs = {}
        for move, p in outcomes:
            nxt = (state[0] + move[0], state[1] + move[1])
            if not self.is_valid_state(nxt):
                nxt = state  # choca con pared/borde -> se queda
            probs[nxt] = probs.get(nxt, 0.0) + p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [2]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                best = max(
                    expected_next_value(grid, s, a, V)
                    for a in grid.actions
                )
                V_new[s] = grid.get_reward(s) + grid.gamma * best

            biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [4]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]  # la politica fija la accion (sin max)
                V_new[s] = (
                    grid.get_reward(s)
                    + grid.gamma * expected_next_value(grid, s, a, V)
                )

            biggest_change = max(biggest_change, abs(V_new[s] - V[s]))

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    new_policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        new_policy[s] = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. politica inicial arbitraria: RIGHT en no terminales.
    policy = {
        s: (0, 1)
        for s in grid.states()
        if not grid.is_terminal(s)
    }

    history = []
    for iteration in range(max_iter):
        V, eval_sweeps = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(new_policy[s] != policy[s] for s in new_policy)
        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_sweeps,
            "changed_actions": changed,
        })

        policy = new_policy
        if changed == 0:   # politica estable -> optima
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)
    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [6]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 73, 'changed_actions': 19}, {'policy_iteration': 2, 'evaluation_sweeps': 61, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


## Experimentos de la Parte 5 (código)

In [7]:
# --- Utilidades para los experimentos de la Parte 5 ---
# Reconstruimos un MDP parametrizable para variar living_reward, gamma
# y la probabilidad del piso resbaloso sin tocar la clase original.

def build_mdp(living_reward=-1.0, gamma=0.9, slippery_intended=0.60):
    g = WarehouseMDP()
    g.living_reward = living_reward
    g.gamma = gamma
    perp = (1.0 - slippery_intended) / 2.0
    g.slippery_probs = (slippery_intended, perp)
    return g

NAMES = {(-1,0):"UP", (1,0):"DOWN", (0,-1):"LEFT", (0,1):"RIGHT"}

def greedy_path_from_start(grid, policy):
    """Sigue el siguiente estado MAS probable desde START hasta un terminal."""
    s = grid.start
    seen = set()
    path = [s]
    for _ in range(60):
        if grid.is_terminal(s):
            break
        a = policy[s]
        nxt = max(grid.get_transition_probs(s, a), key=lambda x: x[1])[0]
        if nxt in seen:
            break
        seen.add(nxt)
        s = nxt
        path.append(s)
    destino = s if grid.is_terminal(s) else "bucle/no-terminal"
    return destino, path

def run_experiment(label, grid):
    V, n = value_iteration(grid)
    pi = extract_policy(grid, V)
    destino, path = greedy_path_from_start(grid, pi)
    print(f"[{label}] iters={n}  V(START)={V[grid.start]:+.3f}  "
          f"START->{NAMES[pi[grid.start]]}  destino_greedy={destino}")
    print("    valores:")
    print_values(grid, V)
    print("    politica:")
    print_policy(grid, pi)
    return V, pi


In [8]:
# === Experimento A — Menos costo por paso (living_reward = -0.1) ===
# Prediccion: con un costo por paso muy pequeno, "caminar" casi no duele,
# asi que compensa recorrer el trayecto largo hasta la ENTREGA +10.
run_experiment("A  living=-0.1", build_mdp(living_reward=-0.1))


[A  living=-0.1] iters=26  V(START)=+1.649  START->RIGHT  destino_greedy=(0, 5)
    valores:
 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874
    politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


({(0, 0): 1.6487099915120942,
  (0, 1): 1.9918824912051392,
  (0, 2): 2.3612510129336486,
  (0, 4): 8.591380001144753,
  (0, 5): 10.0,
  (1, 0): 1.3576414056768187,
  (1, 2): 2.796740887847703,
  (1, 3): 3.910584724087124,
  (1, 4): 4.550397802071998,
  (1, 5): 8.591380001144753,
  (2, 0): 1.2589735578499313,
  (2, 1): 1.5332627286389537,
  (2, 2): 2.0,
  (2, 3): 3.3063598183349074,
  (2, 5): 7.537382198821147,
  (3, 0): 1.2433360250412917,
  (3, 1): 1.5404649507033141,
  (3, 2): 2.0397750004379462,
  (3, 3): 2.417279475261259,
  (3, 4): 2.0260698275842204,
  (3, 5): -10.0,
  (4, 0): 0.8652611152725412,
  (4, 1): -1.7940513080080718,
  (4, 3): 2.0260698275842204,
  (4, 4): 1.7091838363264598,
  (4, 5): 0.8737257368004813},
 {(0, 0): (0, 1),
  (0, 1): (0, 1),
  (0, 2): (1, 0),
  (0, 4): (0, 1),
  (1, 0): (-1, 0),
  (1, 2): (0, 1),
  (1, 3): (0, 1),
  (1, 4): (0, 1),
  (1, 5): (-1, 0),
  (2, 0): (0, 1),
  (2, 1): (0, 1),
  (2, 3): (-1, 0),
  (2, 5): (-1, 0),
  (3, 0): (0, 1),
  (3, 1): (

In [9]:
# === Experimento B — Piso muy resbaloso (0.60 -> 0.40) ===
# Prediccion: con mas ruido en el piso resbaloso el robot se vuelve mas
# conservador cerca de esas celdas, pero el objetivo global apenas cambia.
run_experiment("B  slip=0.40", build_mdp(slippery_intended=0.40))


[B  slip=0.40] iters=22  V(START)=-2.706  START->RIGHT  destino_greedy=(2, 2)
    valores:
 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041
    politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


({(0, 0): -2.706200866141053,
  (0, 1): -1.808762115970734,
  (0, 2): -0.7974981248457793,
  (0, 4): 7.607463797149046,
  (0, 5): 10.0,
  (1, 0): -2.651644711285377,
  (1, 2): 0.3947945647712201,
  (1, 3): 2.1044445680423443,
  (1, 4): 3.6695094728384543,
  (1, 5): 7.607463797149046,
  (2, 0): -1.744435451953327,
  (2, 1): -0.6700085178488235,
  (2, 2): 2.0,
  (2, 3): 0.832041989465881,
  (2, 5): 5.672577665591895,
  (3, 0): -1.8311606302726529,
  (3, 1): -0.7744631315014789,
  (3, 2): 0.5339697103712644,
  (3, 3): -1.1373077117873183,
  (3, 4): -2.1518820834140637,
  (3, 5): -10.0,
  (4, 0): -2.7854148064518496,
  (4, 1): -3.9294880089864095,
  (4, 3): -2.1518820834140637,
  (4, 4): -2.9736967343205816,
  (4, 5): -4.040552060738303},
 {(0, 0): (0, 1),
  (0, 1): (0, 1),
  (0, 2): (1, 0),
  (0, 4): (0, 1),
  (1, 0): (1, 0),
  (1, 2): (1, 0),
  (1, 3): (0, 1),
  (1, 4): (0, 1),
  (1, 5): (-1, 0),
  (2, 0): (0, 1),
  (2, 1): (0, 1),
  (2, 3): (-1, 0),
  (2, 5): (-1, 0),
  (3, 0): (0, 1),


In [10]:
# === Experimento C — Mas paciencia (gamma = 0.99) ===
# Prediccion: con gamma alto las recompensas lejanas pesan mas, de modo
# que la lejana ENTREGA +10 se vuelve mas atractiva que la cercana CARGA +2.
run_experiment("C  gamma=0.99", build_mdp(gamma=0.99))


[C  gamma=0.99] iters=24  V(START)=-1.456  START->RIGHT  destino_greedy=(0, 5)
    valores:
 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892
    politica:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


({(0, 0): -1.4560649388739269,
  (0, 1): -0.309881297820922,
  (0, 2): 0.8089874348550641,
  (0, 4): 8.600762870151303,
  (0, 5): 10.0,
  (1, 0): -2.179072071671536,
  (1, 2): 2.0025662902238364,
  (1, 3): 4.118499537689822,
  (1, 4): 5.35404258745535,
  (1, 5): 8.600762870151303,
  (2, 0): -1.0811884463199115,
  (2, 1): 0.11914882034648855,
  (2, 2): 2.0,
  (2, 3): 2.9127649531468105,
  (2, 5): 7.3954269892381745,
  (3, 0): -1.6057303350535532,
  (3, 1): -0.4668775319202584,
  (3, 2): 0.7993463093272035,
  (3, 3): 0.8173197870576627,
  (3, 4): -0.3592356205701237,
  (3, 5): -10.0,
  (4, 0): -2.7519242206332173,
  (4, 1): -3.737203164683393,
  (4, 3): -0.3592356205701237,
  (4, 4): -1.4075578104698176,
  (4, 5): -2.8923493772381956},
 {(0, 0): (0, 1),
  (0, 1): (0, 1),
  (0, 2): (1, 0),
  (0, 4): (0, 1),
  (1, 0): (1, 0),
  (1, 2): (0, 1),
  (1, 3): (0, 1),
  (1, 4): (0, 1),
  (1, 5): (-1, 0),
  (2, 0): (0, 1),
  (2, 1): (0, 1),
  (2, 3): (-1, 0),
  (2, 5): (-1, 0),
  (3, 0): (0, 1),
 

In [11]:
# === Bonus — Umbral de living_reward donde cambia la meta desde START ===
print("living_reward | destino_greedy desde START")
for lr in [-0.1,-0.3,-0.5,-0.6,-0.7,-0.75,-0.8,-0.9,-1.0,-1.5,-2.0]:
    g = build_mdp(living_reward=lr)
    V, _ = value_iteration(g)
    pi = extract_policy(g, V)
    destino, _ = greedy_path_from_start(g, pi)
    print(f"  {lr:+.2f}        {destino}")
print("\\nEl cambio ENTREGA(+10) -> CARGA(+2) ocurre entre -0.75 y -0.80.")


living_reward | destino_greedy desde START
  -0.10        (0, 5)
  -0.30        (0, 5)
  -0.50        (0, 5)
  -0.60        (0, 5)
  -0.70        (0, 5)
  -0.75        (0, 5)
  -0.80        (2, 2)
  -0.90        (2, 2)
  -1.00        (2, 2)
  -1.50        (2, 2)
  -2.00        (2, 2)
\nEl cambio ENTREGA(+10) -> CARGA(+2) ocurre entre -0.75 y -0.80.


## Parte 5 — Respuestas e interpretación

> Las respuestas se basan en los valores y políticas calculados arriba
> (Value Iteration y Policy Iteration coinciden en la política óptima).

**1. Desde `START`, ¿el robot busca la ENTREGA +10 o la CARGA +2?**
Depende del costo por paso. Con los parámetros base (`living_reward = -1`,
`γ = 0.9`), la política apunta hacia la zona central y el **camino más
probable desde START termina en la CARGA +2 (2,2)**: la entrega está lejos
(≈6–7 pasos) y cada paso cuesta −1, además de bordear peligros (−3, −10),
por lo que el valor descontado de la entrega no compensa el costo acumulado.
La CARGA, más cercana, resulta la mejor meta alcanzable. Si abaratamos el
paso (Exp. A) o somos más pacientes (Exp. C), la meta pasa a ser la
**ENTREGA +10**.

**2. ¿Por qué una recompensa menor (+2) podría ser óptima?**
Porque la *utilidad* no es solo la recompensa final, sino la suma
**descontada** de recompensas menos el costo del camino:
$V(s)=\sum \gamma^t R$. Un premio grande pero lejano se descuenta con
$\gamma^t$ y, sobre todo, acumula muchos costos de paso (−1) y riesgo de
caer en −3/−10. Un premio modesto pero cercano y seguro puede dar mayor
utilidad total. Es el clásico compromiso *recompensa vs. costo/riesgo/distancia*.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**
En las celdas resbalosas `(1,2)`, `(2,1)` y `(3,3)` y sus vecinas. Ahí la
acción deseada solo se cumple con probabilidad 0.6 (vs. 0.9 en piso normal),
con 0.2 de desviarse a cada lado. Eso hace que el agente **evite orientar
movimientos resbalosos hacia celdas peligrosas** (p. ej. cerca de la
`-10` en (3,5) o la `-3` en (1,4)) y prefiera rutas donde una desviación no
sea catastrófica. En el Exp. B (piso 0.40) este efecto se acentúa.

**4. ¿Qué papel cumple el costo por paso −1?**
Es la "prisa" del agente: penaliza cada movimiento, así que incentiva
caminos cortos y castiga dar vueltas. Es justamente lo que inclina la
balanza hacia la CARGA cercana en el caso base; al reducirlo a −0.1 (Exp. A)
el agente puede "permitirse" el viaje largo a la ENTREGA.

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas
probabilidades para todos los estados?**
Porque la dinámica **depende del tipo de piso del estado actual**: piso
normal usa (0.90, 0.05, 0.05) y piso resbaloso (0.60, 0.20, 0.20). Por eso
`get_transition_probs` consulta si `state` está en `slippery_states` y elige
la distribución correspondiente; la transición dejó de ser homogénea.

### Resumen de los experimentos

| Experimento | Cambio | Meta desde START | Lectura |
|---|---|---|---|
| Base | living=−1, γ=0.9 | **CARGA +2** | el costo del viaje no compensa la entrega lejana |
| A | living=−0.1 | **ENTREGA +10** | caminar casi gratis ⇒ vale la pena el premio grande |
| B | slip 0.60→0.40 | CARGA +2 | más ruido ⇒ más cautela, pero la meta global no cambia |
| C | γ=0.99 | **ENTREGA +10** | más paciencia ⇒ el premio lejano pesa más |

**Bonus.** Barriendo `living_reward` con γ=0.9, la meta más probable desde
START cambia de **ENTREGA (+10)** a **CARGA (+2)** aproximadamente entre
`living_reward ≈ -0.75` y `-0.80`. Para costos por paso más baratos que
≈ −0.75 conviene la entrega; para costos más caros, conviene la carga.
